# TDE 03 — Baseline de Previsão
## Análise de Volatilidade de Ações — Optiver Realized Volatility Prediction

**Tema:** Previsão da volatilidade realizada de ações
**Base:** Optiver Realized Volatility Prediction — Kaggle
**Etapa anterior:** TDE 01 — Análise Exploratória de Dados (`TDE01_Analise_Volatilidade_Acoes_Optiver_corrigido.ipynb`)

---

### Objetivo desta etapa

Construir o **baseline** do projeto: modelos de regressão vistos em sala, **sem otimização de hiperparâmetros**, treinados sobre um conjunto simples de atributos. Os resultados servirão como **ponto de comparação** para a próxima etapa, em que os modelos serão otimizados.

Algoritmos utilizados:

| Algoritmo | Classe do scikit-learn | Por que entra no baseline |
|---|---|---|
| Regressão Linear | `LinearRegression` | Modelo mais simples visto em sala; mostra o quanto uma relação linear já explica |
| Árvore de Decisão | `DecisionTreeRegressor` | Captura relações não lineares; permite discutir overfitting |
| Random Forest | `RandomForestRegressor` | Média de várias árvores; tende a generalizar melhor que uma árvore única |

Métricas de avaliação (vistas em sala): **MAE**, **MSE**, **RMSE** e **R²**. Também reportamos o **RMSPE**, métrica oficial da competição no Kaggle, apenas como referência.

## 1. Do TDE 01 para o baseline

A EDA do TDE 01 mostrou que:

1. o `train.csv` tem apenas `stock_id`, `time_id` e `target` — não há atributos prontos para prever o alvo;
2. a volatilidade é **assimétrica** e tem valores extremos legítimos (períodos de estresse), que **não** foram removidos;
3. existem **diferenças entre ações** e **janelas de estresse compartilhadas** entre ações (mesmo `time_id`);
4. variáveis de **microestrutura** (WAP, spread, volumes, trades) mostraram associação com a volatilidade.

Por isso, o baseline segue o caminho indicado nos próximos passos do TDE 01:

> **agregar book + trades → criar atributos simples → separar treino/teste sem vazamento → treinar regressores → avaliar com as métricas da disciplina.**

### Como o problema é formulado

Cada linha do `train.csv` é uma combinação **ação + janela** (`stock_id`, `time_id`). Os dados de *order book* e *trades* descrevem **10 minutos de negociação** dessa janela, e o `target` é a volatilidade realizada dos **10 minutos seguintes**. Portanto:

- **entradas (X):** atributos calculados dos 10 minutos observados;
- **saída (y):** `target`, a volatilidade futura;
- **tipo de problema:** **regressão** (alvo numérico contínuo).

Como os atributos vêm do período anterior ao alvo, não há uso de informação futura.

## 2. Preparação do ambiente

Execute as células em ordem no **Google Colab**. O processo de download é o mesmo do TDE 01:

1. entre na sua conta do Kaggle e aceite as regras da competição **Optiver Realized Volatility Prediction**;
2. gere o arquivo `kaggle.json` (Kaggle → *Settings* → *API* → *Create New Token*);
3. execute a célula de download e envie o `kaggle.json` quando solicitado.

Se os dados já estiverem extraídos em `DATA_DIR`, o download é ignorado.

> **Tempo estimado no Colab gratuito:** download e extração ≈ 5–10 min; cálculo dos atributos das 112 ações ≈ 5–10 min; treino dos modelos ≈ 5 min.

In [ ]:
%pip -q install kaggle pyarrow

In [ ]:
# Configuração da pasta de dados e download (somente se ainda não existir)
import os, glob, zipfile, shutil, subprocess

DATA_DIR = os.environ.get("OPTIVER_DIR", "/content/optiver")
os.makedirs(DATA_DIR, exist_ok=True)

def encontrar(padrao):
    return glob.glob(os.path.join(DATA_DIR, "**", padrao), recursive=True)

if encontrar("train.csv"):
    print("Dados já disponíveis em", DATA_DIR, "- download ignorado.")
else:
    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)
    kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")

    try:
        from google.colab import files
        uploaded = files.upload()  # selecione kaggle.json
        if "kaggle.json" not in uploaded:
            raise FileNotFoundError("Envie o arquivo kaggle.json gerado na sua conta do Kaggle.")
        shutil.copy("kaggle.json", kaggle_json_path)
    except ImportError:
        tem_env = os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")
        if not os.path.exists(kaggle_json_path) and not tem_env:
            raise FileNotFoundError(
                f"Fora do Colab: coloque o kaggle.json em {kaggle_json_path} "
                "ou defina KAGGLE_USERNAME e KAGGLE_KEY."
            )

    if os.path.exists(kaggle_json_path):
        os.chmod(kaggle_json_path, 0o600)

    subprocess.run(
        ["kaggle", "competitions", "download", "-c",
         "optiver-realized-volatility-prediction", "-p", DATA_DIR],
        check=True,
    )

    for z in glob.glob(os.path.join(DATA_DIR, "*.zip")):
        with zipfile.ZipFile(z, "r") as zip_ref:
            zip_ref.extractall(DATA_DIR)
        os.remove(z)  # libera espaço em disco

    print("Download e extração concluídos.")

## 3. Bibliotecas

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from joblib import Parallel, delayed

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")
print("Bibliotecas carregadas.")

## 4. Carregamento do conjunto principal

In [ ]:
train_path = encontrar("train.csv")[0]
train = pd.read_csv(train_path)

print("Arquivo:", train_path)
print(f"Linhas: {train.shape[0]:,} | Colunas: {train.shape[1]}")
print(f"Ações (stock_id): {train['stock_id'].nunique()} | Janelas (time_id): {train['time_id'].nunique()}")
display(train.head())
display(train["target"].describe().to_frame("target"))

## 5. Engenharia de atributos (simples, sem otimização)

Para cada combinação `stock_id` + `time_id`, calculamos atributos dos 10 minutos observados, usando as mesmas medidas do TDE 01.

**Do order book (`book_train.parquet`):**

| Atributo | Definição |
|---|---|
| `rv_wap` | Volatilidade realizada do WAP: $\sqrt{\sum r_t^2}$, com $r_t = \ln(WAP_t) - \ln(WAP_{t-1})$ |
| `rv_wap_ultimos_300s` | A mesma medida, apenas nos últimos 5 minutos da janela |
| `spread_medio` | Média de `ask_price1 - bid_price1` |
| `volume_medio_book` | Média da soma dos volumes dos dois primeiros níveis do book |
| `n_atualizacoes_book` | Quantidade de atualizações do book na janela |

**Dos negócios (`trade_train.parquet`):**

| Atributo | Definição |
|---|---|
| `rv_trade` | Volatilidade realizada calculada com o preço dos negócios |
| `volume_trades` | Soma das quantidades negociadas |
| `ordens_trades` | Soma do número de ordens |
| `n_trades` | Quantidade de registros de negócio |

Onde: $WAP = \dfrac{bid\_price_1 \cdot ask\_size_1 + ask\_price_1 \cdot bid\_size_1}{bid\_size_1 + ask\_size_1}$.

Janelas sem negócios recebem 0 nos atributos de trades. O `stock_id` **não** é usado como atributo numérico, pois é uma variável categórica nominal (TDE 01): tratá-lo como número faria o modelo supor que a ação 100 é "maior" que a ação 2. Codificar as ações fica para a etapa de otimização.

Os arquivos Parquet são lidos **uma ação por vez**, evitando o problema de memória apontado nas limitações do TDE 01. O resultado é salvo em CSV para não ser recalculado.

In [ ]:
def stock_id_do_caminho(caminho):
    # As partições têm o formato .../stock_id=<n>
    return int(os.path.basename(caminho.rstrip("/")).split("=")[1])


def atributos_book(caminho):
    book = pd.read_parquet(caminho).sort_values(["time_id", "seconds_in_bucket"])

    book["wap"] = (
        book["bid_price1"] * book["ask_size1"] + book["ask_price1"] * book["bid_size1"]
    ) / (book["bid_size1"] + book["ask_size1"])
    book["log_return"] = np.log(book["wap"]).groupby(book["time_id"]).diff()
    book["r2"] = book["log_return"] ** 2
    book["spread"] = book["ask_price1"] - book["bid_price1"]
    book["volume_total"] = (
        book["bid_size1"] + book["ask_size1"] + book["bid_size2"] + book["ask_size2"]
    )

    g = book.groupby("time_id")
    feats = pd.DataFrame({
        "rv_wap": np.sqrt(g["r2"].sum()),
        "spread_medio": g["spread"].mean(),
        "volume_medio_book": g["volume_total"].mean(),
        "n_atualizacoes_book": g.size(),
    })
    ultimos = book[book["seconds_in_bucket"] >= 300].groupby("time_id")["r2"].sum()
    feats["rv_wap_ultimos_300s"] = np.sqrt(ultimos).reindex(feats.index).fillna(0)
    return feats


def atributos_trade(caminho):
    trade = pd.read_parquet(caminho).sort_values(["time_id", "seconds_in_bucket"])
    trade["log_return"] = np.log(trade["price"]).groupby(trade["time_id"]).diff()
    trade["r2"] = trade["log_return"] ** 2

    g = trade.groupby("time_id")
    return pd.DataFrame({
        "rv_trade": np.sqrt(g["r2"].sum()),
        "volume_trades": g["size"].sum(),
        "ordens_trades": g["order_count"].sum(),
        "n_trades": g.size(),
    })


def atributos_acao(caminho_book):
    stock_id = stock_id_do_caminho(caminho_book)
    feats = atributos_book(caminho_book)

    caminho_trade = caminho_book.replace("book_train.parquet", "trade_train.parquet")
    colunas_trade = ["rv_trade", "volume_trades", "ordens_trades", "n_trades"]
    if os.path.exists(caminho_trade):
        feats = feats.join(atributos_trade(caminho_trade), how="left")
    else:
        for c in colunas_trade:
            feats[c] = np.nan
    feats[colunas_trade] = feats[colunas_trade].fillna(0)

    feats = feats.reset_index()
    feats.insert(0, "stock_id", stock_id)
    return feats

In [ ]:
CACHE_FEATURES = os.path.join(DATA_DIR, "features_baseline.csv")

if os.path.exists(CACHE_FEATURES):
    features = pd.read_csv(CACHE_FEATURES)
    print("Atributos carregados do cache:", CACHE_FEATURES)
else:
    pastas_book = sorted(
        p for p in glob.glob(os.path.join(DATA_DIR, "**", "book_train.parquet", "stock_id=*"), recursive=True)
        if os.path.isdir(p)
    )
    print("Partições de ações encontradas:", len(pastas_book))

    inicio = time.time()
    partes = Parallel(n_jobs=-1, verbose=5)(delayed(atributos_acao)(p) for p in pastas_book)
    features = pd.concat(partes, ignore_index=True)
    features.to_csv(CACHE_FEATURES, index=False)
    print(f"Atributos calculados em {time.time() - inicio:.1f} s e salvos em {CACHE_FEATURES}")

print("Formato:", features.shape)
display(features.head())

### Junção com o alvo e verificação de qualidade

In [ ]:
dados = train.merge(features, on=["stock_id", "time_id"], how="inner")

print(f"Linhas do train.csv: {len(train):,}")
print(f"Linhas após a junção: {len(dados):,} ({len(dados) / len(train) * 100:.2f}% do train.csv)")
print("Valores ausentes nos atributos:", int(dados.isna().sum().sum()))
print("Valores infinitos nos atributos:", int(np.isinf(dados.select_dtypes("number")).sum().sum()))

ATRIBUTOS = [
    "rv_wap", "rv_wap_ultimos_300s", "spread_medio", "volume_medio_book",
    "n_atualizacoes_book", "rv_trade", "volume_trades", "ordens_trades", "n_trades",
]
display(dados[ATRIBUTOS + ["target"]].describe().T)

In [ ]:
# Correlação de cada atributo com o alvo (associação, não causalidade)
corr_alvo = dados[ATRIBUTOS + ["target"]].corr()["target"].drop("target").sort_values(key=abs, ascending=False)
display(corr_alvo.to_frame("correlação com target"))

plt.figure(figsize=(8, 5))
cores = ["tab:blue" if v >= 0 else "tab:red" for v in corr_alvo.values[::-1]]
plt.barh(corr_alvo.index[::-1], corr_alvo.values[::-1], color=cores)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Correlação de Pearson entre atributos e volatilidade futura")
plt.xlabel("Correlação")
plt.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

## 6. Separação em treino e teste

Usamos **80% para treino** e **20% para teste**, com `random_state=42` para reprodutibilidade.

**Cuidado contra vazamento:** o TDE 01 mostrou que várias ações ficam voláteis **ao mesmo tempo** (picos por `time_id`). Se uma mesma janela tivesse ações no treino e outras no teste, o modelo "veria" o estresse daquele momento durante o treino, e o teste ficaria otimista. Por isso, a divisão é feita **por `time_id`**: todas as ações de uma janela ficam inteiramente no treino ou inteiramente no teste.

O conjunto de teste é usado **somente** para a avaliação final; nenhum hiperparâmetro foi escolhido olhando para ele.

In [ ]:
time_ids = dados["time_id"].unique()
tid_treino, tid_teste = train_test_split(time_ids, test_size=0.20, random_state=SEED)

mask_treino = dados["time_id"].isin(tid_treino)
treino, teste = dados[mask_treino], dados[~mask_treino]

X_train, y_train = treino[ATRIBUTOS], treino["target"]
X_test, y_test = teste[ATRIBUTOS], teste["target"]

print(f"time_id no treino: {len(tid_treino):,} | no teste: {len(tid_teste):,}")
print(f"Linhas de treino: {len(X_train):,} | Linhas de teste: {len(X_test):,}")
print("Janelas em comum entre treino e teste:", len(set(treino["time_id"]) & set(teste["time_id"])))

## 7. Métricas de avaliação

| Métrica | Fórmula | Leitura |
|---|---|---|
| **MAE** | $\frac{1}{n}\sum \lvert y - \hat{y} \rvert$ | Erro absoluto médio, na unidade do alvo. Métrica principal indicada para o projeto |
| **MSE** | $\frac{1}{n}\sum (y - \hat{y})^2$ | Penaliza mais os erros grandes |
| **RMSE** | $\sqrt{MSE}$ | Mesma ideia do MSE, de volta à unidade do alvo |
| **R²** | $1 - \frac{\sum (y-\hat{y})^2}{\sum (y-\bar{y})^2}$ | Proporção da variação explicada; 0 = igual a prever a média |
| **RMSPE** | $\sqrt{\frac{1}{n}\sum \left(\frac{y-\hat{y}}{y}\right)^2}$ | Erro percentual, métrica oficial da competição (referência) |

Para todas, exceto o R², **menor é melhor**. As métricas são calculadas no **treino** e no **teste**: a diferença entre elas revela overfitting (erro baixo no treino e alto no teste).

In [ ]:
def rmspe(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(np.sqrt(np.mean(((y_true - y_pred) / y_true) ** 2)))


def avaliar(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred),
        "RMSPE": rmspe(y_true, y_pred),
    }

## 8. Referências mínimas (não são os algoritmos do baseline)

Antes dos modelos, medimos duas referências que **não aprendem nada** dos atributos. Um modelo só é útil se superar ambas.

- **Média do treino** (`DummyRegressor`): prevê sempre a volatilidade média do treino.
- **Persistência ingênua:** prevê que a volatilidade dos próximos 10 minutos será igual à dos 10 minutos observados (`previsão = rv_wap`). É a referência clássica desta competição.

In [ ]:
resultados = []

dummy = DummyRegressor(strategy="mean").fit(X_train, y_train)
resultados.append({"modelo": "Referência: média do treino", "conjunto": "treino", **avaliar(y_train, dummy.predict(X_train))})
resultados.append({"modelo": "Referência: média do treino", "conjunto": "teste", **avaliar(y_test, dummy.predict(X_test))})

resultados.append({"modelo": "Referência: persistência (rv_wap)", "conjunto": "treino", **avaliar(y_train, X_train["rv_wap"])})
resultados.append({"modelo": "Referência: persistência (rv_wap)", "conjunto": "teste", **avaliar(y_test, X_test["rv_wap"])})

display(pd.DataFrame(resultados))

## 9. Treinamento do baseline

Os três algoritmos são treinados com os **valores padrão** do scikit-learn, alterando apenas `random_state` (reprodutibilidade) e `n_jobs` (uso de todos os núcleos, sem efeito no resultado). Não há busca de hiperparâmetros, validação cruzada nem seleção de atributos: isso é exatamente o que será feito na próxima etapa e comparado com este baseline.

**Padronização na Regressão Linear.** Os atributos têm escalas muito diferentes (volatilidades na casa de 0,001 e volumes na casa das centenas). Isso deixa o cálculo da reta numericamente instável, e o modelo pode acabar prevendo quase só a média. Por isso a Regressão Linear é precedida de um `StandardScaler` dentro de um *pipeline*: a média e o desvio-padrão são aprendidos **apenas no treino** (`fit`) e somente aplicados no teste (`transform`), sem vazamento. Isso é pré-processamento, não otimização. As árvores não precisam de padronização, pois dividem os dados por limiares em cada atributo.

In [ ]:
modelos = {
    "Regressão Linear": make_pipeline(StandardScaler(), LinearRegression()),
    "Árvore de Decisão": DecisionTreeRegressor(random_state=SEED),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1),
}

previsoes_teste = {}
tempos = {}

for nome, modelo in modelos.items():
    inicio = time.time()
    modelo.fit(X_train, y_train)
    tempos[nome] = time.time() - inicio

    pred_treino = modelo.predict(X_train)
    pred_teste = modelo.predict(X_test)
    previsoes_teste[nome] = pred_teste

    resultados.append({"modelo": nome, "conjunto": "treino", **avaliar(y_train, pred_treino)})
    resultados.append({"modelo": nome, "conjunto": "teste", **avaliar(y_test, pred_teste)})
    print(f"{nome:<20} treinado em {tempos[nome]:6.1f} s")

### Registro das configurações utilizadas

Registrar os hiperparâmetros é parte do baseline: sem eles a comparação com a etapa de otimização não seria reproduzível.

In [ ]:
registro = pd.DataFrame([
    {"modelo": "Regressão Linear", "hiperparâmetros": "StandardScaler + LinearRegression padrão (fit_intercept=True)", "tempo de treino (s)": tempos["Regressão Linear"]},
    {"modelo": "Árvore de Decisão", "hiperparâmetros": "padrão (max_depth=None, min_samples_leaf=1), random_state=42", "tempo de treino (s)": tempos["Árvore de Decisão"]},
    {"modelo": "Random Forest", "hiperparâmetros": "padrão (n_estimators=100, max_depth=None, max_features=1.0), random_state=42", "tempo de treino (s)": tempos["Random Forest"]},
])
registro["profundidade / folhas"] = [
    "-",
    f"{modelos['Árvore de Decisão'].get_depth()} níveis / {modelos['Árvore de Decisão'].get_n_leaves():,} folhas",
    f"média de {np.mean([a.get_depth() for a in modelos['Random Forest'].estimators_]):.0f} níveis por árvore",
]
display(registro)

## 10. Resultados

In [ ]:
tabela = pd.DataFrame(resultados)
tabela_teste = tabela[tabela["conjunto"] == "teste"].drop(columns="conjunto").set_index("modelo").sort_values("MAE")

print("=== Desempenho no conjunto de TESTE (ordenado pelo MAE) ===")
display(tabela_teste.style.format("{:.6f}").highlight_min(subset=["MAE", "MSE", "RMSE", "RMSPE"], color="#c6efce")
        .highlight_max(subset=["R2"], color="#c6efce"))

In [ ]:
# Treino x teste: diagnóstico de overfitting
comparacao = tabela.pivot(index="modelo", columns="conjunto", values=["MAE", "R2"])
comparacao.columns = [f"{m} ({c})" for m, c in comparacao.columns]
comparacao["MAE teste / MAE treino"] = comparacao["MAE (teste)"] / comparacao["MAE (treino)"]
comparacao = comparacao.loc[tabela_teste.index]
display(comparacao)

nomes_modelos = list(modelos.keys())
mae_treino = [comparacao.loc[n, "MAE (treino)"] for n in nomes_modelos]
mae_teste = [comparacao.loc[n, "MAE (teste)"] for n in nomes_modelos]
x = np.arange(len(nomes_modelos))

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - 0.2, mae_treino, width=0.4, label="Treino", color="tab:gray")
ax.bar(x + 0.2, mae_teste, width=0.4, label="Teste", color="tab:blue")
ax.axhline(comparacao.loc["Referência: persistência (rv_wap)", "MAE (teste)"], color="tab:red",
           linestyle="--", label="Persistência ingênua (teste)")
ax.set_xticks(x)
ax.set_xticklabels(nomes_modelos)
ax.set_ylabel("MAE")
ax.set_title("MAE no treino e no teste por algoritmo")
ax.legend()
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Valores reais x previstos no teste (amostra de até 5.000 pontos)
amostra = np.random.RandomState(SEED).choice(len(y_test), size=min(5000, len(y_test)), replace=False)
y_amostra = y_test.to_numpy()[amostra]
limite = np.quantile(y_test, 0.995)

fig, axes = plt.subplots(1, len(modelos), figsize=(15, 5), sharex=True, sharey=True)
for ax, nome in zip(axes, modelos):
    ax.scatter(y_amostra, previsoes_teste[nome][amostra], s=5, alpha=0.3)
    ax.plot([0, limite], [0, limite], color="crimson", label="Previsão perfeita")
    ax.set_xlim(0, limite)
    ax.set_ylim(0, limite)
    ax.set_title(f"{nome}\nMAE = {tabela_teste.loc[nome, 'MAE']:.6f}")
    ax.set_xlabel("Volatilidade real")
    ax.grid(alpha=0.2)
axes[0].set_ylabel("Volatilidade prevista")
axes[0].legend()
plt.suptitle("Teste: valores reais x previstos (eixos cortados no percentil 99,5%)")
plt.tight_layout()
plt.show()

In [ ]:
# Distribuição dos erros no teste
plt.figure(figsize=(10, 5))
for nome in modelos:
    erro = y_test.to_numpy() - previsoes_teste[nome]
    plt.hist(erro, bins=120, range=np.quantile(erro, [0.005, 0.995]), alpha=0.45, label=nome)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Distribuição do erro (real - previsto) no teste")
plt.xlabel("Erro")
plt.ylabel("Frequência")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# O que cada modelo considerou importante
importancia = pd.DataFrame({
    "Random Forest (importância)": modelos["Random Forest"].feature_importances_,
    "Árvore de Decisão (importância)": modelos["Árvore de Decisão"].feature_importances_,
    "Regressão Linear (coef. padronizado)": modelos["Regressão Linear"][-1].coef_,
}, index=ATRIBUTOS).sort_values("Random Forest (importância)", ascending=False)
display(importancia)

imp_rf = importancia["Random Forest (importância)"].sort_values()
plt.figure(figsize=(8, 5))
plt.barh(imp_rf.index, imp_rf.values, color="tab:green")
plt.title("Importância dos atributos - Random Forest")
plt.xlabel("Importância (redução média de erro)")
plt.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

## 11. Discussão dos resultados

A célula abaixo gera a leitura dos resultados **a partir dos números obtidos**, para que o texto sempre corresponda à execução.

In [ ]:
mae_t = tabela_teste["MAE"]
r2_t = tabela_teste["R2"]
melhor = mae_t[nomes_modelos].idxmin()
pior = mae_t[nomes_modelos].idxmax()
mae_persist = mae_t["Referência: persistência (rv_wap)"]
mae_media = mae_t["Referência: média do treino"]

def ganho(a, b):
    return (b - a) / b * 100

print("=== LEITURA DOS RESULTADOS ===\n")
print(f"1) Melhor algoritmo no teste pelo MAE: {melhor} "
      f"(MAE = {mae_t[melhor]:.6f}, RMSE = {tabela_teste.loc[melhor, 'RMSE']:.6f}, R² = {r2_t[melhor]:.4f}).")
print(f"   Isso significa que, em média, a volatilidade prevista erra {mae_t[melhor]:.6f} "
      f"para mais ou para menos; a volatilidade média do teste é {y_test.mean():.6f}.")
print(f"   Pior algoritmo no teste: {pior} (MAE = {mae_t[pior]:.6f}).\n")

print("2) Comparação com as referências (MAE no teste):")
for nome in nomes_modelos:
    print(f"   - {nome:<18}: {ganho(mae_t[nome], mae_media):+6.1f}% vs média | "
          f"{ganho(mae_t[nome], mae_persist):+6.1f}% vs persistência  (positivo = melhor)")
print()

print("3) Treino x teste (overfitting):")
for nome in nomes_modelos:
    tr = comparacao.loc[nome, "MAE (treino)"]
    te = comparacao.loc[nome, "MAE (teste)"]
    razao = te / tr if tr > 0 else np.inf
    txt_razao = f"{razao:.1f}x" if np.isfinite(razao) else "treino com erro zero"
    if razao > 2:
        leitura = "erro de teste muito maior que o de treino -> OVERFITTING (decorou o ruído do treino)"
    elif razao > 1.2:
        leitura = "diferença moderada -> algum overfitting"
    else:
        leitura = "treino e teste próximos -> generaliza de forma estável"
    print(f"   - {nome:<18}: MAE treino {tr:.6f} | teste {te:.6f} | razão {txt_razao} -> {leitura}")
print()

print("4) Atributo mais relevante para o Random Forest:", importancia.index[0],
      f"({importancia.iloc[0, 0] * 100:.1f}% da importância total).")
print(f"   Correlação desse atributo com o alvo: {corr_alvo[importancia.index[0]]:.3f}.")

### Interpretação

**Por que o baseline é avaliado assim.** O MAE mede o erro médio na própria unidade da volatilidade, por isso é a métrica principal da comparação. O RMSE, por elevar os erros ao quadrado, é mais sensível às janelas de estresse — a cauda direita identificada no TDE 01 — e a diferença entre RMSE e MAE indica o quanto os erros grandes pesam. O R² compara o modelo com o simples "prever a média".

**Regressão Linear.** Como a volatilidade passada (`rv_wap`) tem forte associação positiva com a volatilidade futura, uma combinação linear dos atributos já captura grande parte do padrão. O modelo é simples, rápido e costuma apresentar erros de treino e teste **próximos**, ou seja, não decora os dados. A limitação é não representar relações não lineares, como o efeito diferente de um spread alto em ações líquidas e ilíquidas.

**Árvore de Decisão.** Com os valores padrão (`max_depth=None`, `min_samples_leaf=1`), a árvore cresce até quase isolar cada exemplo. O resultado esperado — e que deve ser confirmado na tabela de treino x teste — é um **erro de treino próximo de zero e um erro de teste bem maior**: o caso clássico de **overfitting** visto em sala. Esse é o modelo com mais espaço de ganho na etapa de otimização (limitar `max_depth`, aumentar `min_samples_leaf`).

**Random Forest.** Ao tirar a média de 100 árvores treinadas em amostras diferentes, o Random Forest reduz a variância de uma árvore isolada. Por isso costuma ter MAE de teste menor que a árvore única, mesmo sem ajuste. Ainda há diferença entre treino e teste, porque cada árvore também é profunda.

**Referências.** Superar a média do treino é obrigatório; superar a **persistência ingênua** mostra que os modelos aprendem algo além de "a volatilidade se repete". A volatilidade realizada tende a ser persistente no curto prazo, o que torna essa referência difícil de vencer por grande margem.

**Atributos.** A importância do Random Forest e os coeficientes padronizados da Regressão Linear (efeito de aumentar o atributo em um desvio-padrão) indicam quais sinais da microestrutura foram usados. Atributos de volatilidade recente (`rv_wap`, `rv_wap_ultimos_300s`, `rv_trade`) tendem a dominar, confirmando o insight 5 do TDE 01. A importância mostra **o que o modelo usou**, não uma relação de causa.

> **Atenção ao entregar:** os parágrafos acima descrevem o comportamento esperado de cada algoritmo. Depois de executar o notebook, confira-os com a saída da célula "LEITURA DOS RESULTADOS" e ajuste qualquer afirmação que os números não confirmem.

## 12. Limitações do baseline

1. **Poucos atributos.** Apenas nove medidas agregadas da janela inteira; não há atributos por sub-janelas, do segundo nível do book (WAP2) nem por ação.
2. **`stock_id` não usado.** O TDE 01 mostrou diferenças persistentes entre ações, que ainda não foram aproveitadas.
3. **Sem ajuste de hiperparâmetros.** Proposital: é isso que o baseline mede.
4. **Uma única divisão treino/teste.** O resultado pode variar com outra divisão; a validação cruzada fica para a próxima etapa.
5. **Função de perda diferente da métrica oficial.** Os modelos minimizam o erro quadrático, enquanto a competição usa erro percentual (RMSPE), que dá mais peso às janelas de baixa volatilidade.

## 13. Próximos passos (etapa de otimização)

O baseline registrado acima será a referência para comparar:

1. **ajuste de hiperparâmetros** com validação cruzada por grupos de `time_id` (`GroupKFold`): `max_depth` e `min_samples_leaf` na árvore; `n_estimators`, `max_depth` e `max_features` no Random Forest;
2. **regularização** (Ridge e Lasso) no modelo linear;
3. **novos atributos:** WAP do segundo nível, volatilidade em sub-janelas (últimos 100/200 s), desequilíbrio entre volume de compra e venda, média da volatilidade por ação;
4. **transformação do alvo** (por exemplo, logaritmo) para lidar com a assimetria observada no TDE 01;
5. avaliar sempre nas **mesmas métricas** e no **mesmo conjunto de teste** usado aqui.

## 14. Conclusão

Este notebook construiu o baseline do projeto de previsão de volatilidade realizada. A partir dos dados de *order book* e *trades*, foram calculados atributos simples de microestrutura para cada ação e janela. Os dados foram divididos em treino e teste por `time_id`, evitando vazamento entre janelas de estresse compartilhadas, e três algoritmos vistos em sala — **Regressão Linear, Árvore de Decisão e Random Forest** — foram treinados sem otimização e avaliados com **MAE, MSE, RMSE e R²**.

Os resultados obtidos, junto com as duas referências ingênuas e o registro das configurações, formam o ponto de partida reproduzível para a próxima etapa, cujo objetivo é reduzir o MAE de teste em relação a este baseline.

### Checklist da atividade

- [x] Projeto do TDE 01 (Optiver Realized Volatility Prediction)
- [x] Pelo menos dois algoritmos vistos em sala (três utilizados)
- [x] Sem otimização de hiperparâmetros
- [x] Métricas de avaliação de modelos (MAE, MSE, RMSE, R²)
- [x] Implementação e discussão dos resultados no notebook